In [12]:
import xarray as xr
import numpy as np
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader

In [2]:

FILES = {
    "ACE": "ace.nc",
    "Wave-Gauss": "wg.nc",
}

for name, path in FILES.items():
    print("\n" + "=" * 60)
    print(f"DATASET: {name}")
    print("=" * 60)

    ds = xr.open_dataset(path)

    print("\nVariables:")
    for var in ds.data_vars:
        print(f"  {var}: shape={ds[var].shape}, dims={ds[var].dims}, dtype={ds[var].dtype}")

    print("\nCoordinates:")
    for coord in ds.coords:
        print(f"  {coord}: shape={ds[coord].shape}, dims={ds[coord].dims}")

    print("\nAttributes:")
    print(ds.attrs)

    ds.close()


DATASET: ACE

Variables:
  solution: shape=(3750, 20, 128, 128), dims=('sample', 'time', 'x', 'y'), dtype=float32

Coordinates:

Attributes:
{}

DATASET: Wave-Gauss

Variables:
  solution: shape=(3504, 16, 128, 128), dims=('sample', 'time', 'x', 'y'), dtype=float32

Coordinates:

Attributes:
{}


In [3]:

FILES = {
    "ACE": "ace.nc",
    "Wave-Gauss": "wg.nc",
}

for name, path in FILES.items():
    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    ds = xr.open_dataset(path)
    data = ds["solution"]

    # Проверяем данные, не загружая весь датасет в RAM
    print("Shape:", data.shape)
    print("Dtype:", data.dtype)

    # Проверяем небольшой фрагмент
    x = data.isel(sample=slice(0, 100)).values

    print("Min:", np.nanmin(x))
    print("Max:", np.nanmax(x))
    print("Mean:", np.nanmean(x))
    print("Std:", np.nanstd(x))

    print("NaN count:", np.isnan(x).sum())
    print("Inf count:", np.isinf(x).sum())

    ds.close()



ACE
Shape: (3750, 20, 128, 128)
Dtype: float32
Min: -0.99986625
Max: 0.9997645
Mean: -0.05397928
Std: 0.6427164
NaN count: 0
Inf count: 0

Wave-Gauss
Shape: (3504, 16, 128, 128)
Dtype: float32
Min: -0.585687
Max: 2.0579555
Mean: 0.03309741
Std: 0.11988654
NaN count: 0
Inf count: 0


In [4]:
SEED = 42
rng = np.random.default_rng(SEED)

CONFIG = {
    "ace": {
        "file": "ace.nc",
        "time_indices": np.linspace(0, 19, 16, dtype=int),
    },
    "wg": {
        "file": "wg.nc",
        "time_indices": np.arange(16),
    },
}

os.makedirs("data", exist_ok=True)

for name, cfg in CONFIG.items():
    print(f"\nPreparing {name}...")

    with xr.open_dataset(
        cfg["file"],
        engine="h5netcdf"
    ) as ds:

        data = ds["solution"]

        indices = rng.choice(
            data.shape[0],
            size=140,
            replace=False
        )

        samples = []

        for i, idx in enumerate(indices):
            sample = data.isel(sample=int(idx)).values
            sample = sample[cfg["time_indices"], :, :]
            samples.append(sample)

            if (i + 1) % 20 == 0:
                print(f"Read {i + 1}/140")

        samples = np.stack(samples).astype(np.float32)

    np.save(f"data/{name}_train.npy", samples[:100])
    np.save(f"data/{name}_val.npy", samples[100:120])
    np.save(f"data/{name}_test.npy", samples[120:140])

    print("Train:", samples[:100].shape)
    print("Validation:", samples[100:120].shape)
    print("Test:", samples[120:140].shape)

print("\nDone!")



Preparing ace...
Read 20/140
Read 40/140
Read 60/140
Read 80/140
Read 100/140
Read 120/140
Read 140/140
Train: (100, 16, 128, 128)
Validation: (20, 16, 128, 128)
Test: (20, 16, 128, 128)

Preparing wg...
Read 20/140
Read 40/140
Read 60/140
Read 80/140
Read 100/140
Read 120/140
Read 140/140
Train: (100, 16, 128, 128)
Validation: (20, 16, 128, 128)
Test: (20, 16, 128, 128)

Done!


In [5]:

FILES = [
    "ace_train.npy", "ace_val.npy", "ace_test.npy",
    "wg_train.npy", "wg_val.npy", "wg_test.npy"
]

for filename in FILES:
    path = os.path.join("data", filename)
    data = np.load(path, mmap_mode="r")

    print(f"\n{filename}")
    print("Shape:", data.shape)
    print("Dtype:", data.dtype)
    print("NaN:", np.isnan(data).sum())
    print("Inf:", np.isinf(data).sum())

    assert data.shape[1:] == (16, 128, 128)
    assert data.dtype == np.float32
    assert np.isfinite(data).all()

print("\nAll datasets passed validation!")



ace_train.npy
Shape: (100, 16, 128, 128)
Dtype: float32
NaN: 0
Inf: 0

ace_val.npy
Shape: (20, 16, 128, 128)
Dtype: float32
NaN: 0
Inf: 0

ace_test.npy
Shape: (20, 16, 128, 128)
Dtype: float32
NaN: 0
Inf: 0

wg_train.npy
Shape: (100, 16, 128, 128)
Dtype: float32
NaN: 0
Inf: 0

wg_val.npy
Shape: (20, 16, 128, 128)
Dtype: float32
NaN: 0
Inf: 0

wg_test.npy
Shape: (20, 16, 128, 128)
Dtype: float32
NaN: 0
Inf: 0

All datasets passed validation!


In [6]:
ace = np.load("data/ace_train.npy", mmap_mode="r")
wg = np.load("data/wg_train.npy", mmap_mode="r")

# Метки классов
ace_labels = np.ones(len(ace), dtype=np.int64)
wg_labels = np.full(len(wg), 2, dtype=np.int64)

# Объединяем метки
labels = np.concatenate([ace_labels, wg_labels])

print("ACE labels:", np.unique(ace_labels))
print("Wave-Gauss labels:", np.unique(wg_labels))

print("Total samples:", len(labels))
print("Class distribution:", np.unique(labels, return_counts=True))

assert len(labels) == 200
assert set(labels) == {1, 2}

print("\nConditions are ready!")


ACE labels: [1]
Wave-Gauss labels: [2]
Total samples: 200
Class distribution: (array([1, 2]), array([100, 100]))

Conditions are ready!


In [10]:
os.makedirs("data/normalized", exist_ok=True)

CLASSES = ["ace", "wg"]
stats = {}

for name in CLASSES:
    print(f"\nNormalizing {name}...")

    train = np.load(f"data/{name}_train.npy")
    val = np.load(f"data/{name}_val.npy")
    test = np.load(f"data/{name}_test.npy")

    # Статистика только по Train
    mean = float(train.mean())
    std = float(train.std())

    print("Mean:", mean)
    print("Std:", std)

    eps = 1e-6

    train_norm = (train - mean) / (std + eps)
    val_norm = (val - mean) / (std + eps)
    test_norm = (test - mean) / (std + eps)

    np.save(f"data/normalized/{name}_train.npy",
            train_norm.astype(np.float32))
    np.save(f"data/normalized/{name}_val.npy",
            val_norm.astype(np.float32))
    np.save(f"data/normalized/{name}_test.npy",
            test_norm.astype(np.float32))

    stats[name] = {
        "mean": mean,
        "std": std
    }

    print("Normalized train mean:", train_norm.mean())
    print("Normalized train std:", train_norm.std())

with open("data/normalized/stats.json", "w") as f:
    json.dump(stats, f, indent=4)

print("\nNormalization completed!")



Normalizing ace...
Mean: 0.020347371697425842
Std: 0.6351273059844971
Normalized train mean: 1.4305114e-08
Normalized train std: 0.9999984

Normalizing wg...
Mean: 0.03335093334317207
Std: 0.11633508652448654
Normalized train mean: 1.66893e-08
Normalized train std: 0.9999914

Normalization completed!


In [13]:

class PDE_Dataset(Dataset):
    def __init__(self, split="train"):
        ace = np.load(f"data/normalized/ace_{split}.npy")
        wg = np.load(f"data/normalized/wg_{split}.npy")

        # Объединяем два класса
        data = np.concatenate([ace, wg], axis=0)

        # Метки классов
        ace_labels = np.ones(len(ace), dtype=np.int64)
        wg_labels = np.full(len(wg), 2, dtype=np.int64)

        labels = np.concatenate([ace_labels, wg_labels])

        self.data = torch.from_numpy(data).float()
        self.labels = torch.from_numpy(labels).long()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


if __name__ == "__main__":
    train_dataset = PDE_Dataset("train")

    train_loader = DataLoader(
        train_dataset,
        batch_size=4,
        shuffle=True
    )

    x, labels = next(iter(train_loader))

    print("Batch shape:", x.shape)
    print("Labels:", labels)
    print("Dataset size:", len(train_dataset))


Batch shape: torch.Size([4, 16, 128, 128])
Labels: tensor([1, 1, 1, 1])
Dataset size: 200
